In [ ]:
# In particular, a research group in Computer Science at UBC, led by Frank Wood, is collecting data about how people play
# video games. They have set up a MineCraft server, and players' actions are recorded as they navigate through the world. 
# But running this project is not simple: they need to target their recruitment efforts, and make sure they have enough resources
# (e.g., software licenses, server hardware) to handle the number of players they attract.

In [ ]:
# submit as .html and as .ipynb
# max 500 words (non-inclusive of code)
# should not require other files (no reading in data, just name variables & etc)

# chosen question (Question 2):
# We would like to know which "kinds" of players are most likely to contribute
# a large amount of data so that we can target those players in our recruiting efforts.

# should only need players.csv to answer this question

In [1]:
# the problem (dissected):
# "contribute a large amount of data" = high played hours
# predicting high played hours based on other qualities = k-nn regression
# player data -> -> k-nn regression

In [2]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


(1) Data Description:
Provide a full descriptive summary of the dataset, including information such as the number of observations, summary statistics (report values to 2 decimal places), number of variables, name and type of variables, what the variables mean, any issues you see in the data, any other potential issues related to things you cannot directly see, how the data were collected, etc. Make sure to use bullet point lists or tables to summarize the variables in an easy-to-understand format.

Note that the selected dataset(s) will probably contain more variables than you need. In fact, exploring how the different variables in the dataset affect your model may be a crucial part of the project. You need to summarize the full data regardless of which variables you may choose to use later on.

**1. Data Description**
The dataset I am using, `players.csv`, has 196 observations with 7 variables. The variables are described in the table below:

In [18]:
variable_desc_table <- data.frame(
    name = c("experience", "subscribe", "hashedEmail", "played_hours", "name", "gender", "Age"),
    type = c("character", "logical", "character", "double", "character", "character", "double"),
    mean_stat_if_dbl = c("NA", "NA", "NA", "5.85", "NA", "NA", "21.14"),
    description = c("The experience level of the player in playing Minecraft (Beginner, Amateur, Regular, Pro, 
                    Veteran)", "If the player is subscribed to the gaming newsletter", "Anonymous representation
                    of player's email address", "Hours played on Minecraft server", "Player's first name", "Player's 
                    gender (Male, Female, Non-binary, Two-Spirited, Agender, Prefer not to say, Other)", "Player's age")
    )
variable_desc_table

name,type,mean_stat_if_dbl,description
<chr>,<chr>,<chr>,<chr>
experience,character,NA,"The experience level of the player in playing Minecraft (Beginner, Amateur, Regular, Pro, Veteran)"
subscribe,logical,NA,If the player is subscribed to the gaming newsletter
hashedEmail,character,NA,Anonymous representation of player's email address
played_hours,double,5.85,Hours played on Minecraft server
name,character,NA,Player's first name
gender,character,NA,"Player's gender (Male, Female, Non-binary, Two-Spirited, Agender, Prefer not to say, Other)"
Age,double,21.14,Player's age


The only issue that comes to mind when dealing with the data is the use of the hashed emails as the only real unique distinctive value between each observation, since they are such long and random character strings that aren't easy for humans to read. While I couldn't see any duplicate player names, it is still possible I missed one given the number of observations, so it is not a good variable to use for this. However, I do not forsee a need to filter for an individual observation often (if at all), so I am not concerned. Additionally, I noticed there are some seemingly odd name and gender pairings, given societal naming conventions, which could be indicative of the data being transferred to the `.csv` file incorrectly. That is the only reason I note it, but there are likely other explanations, so I assume it to be a non-issue in the meantime. Lastly, some observations have `NA` in some variables, but that is easy to resolve with filtration. 

I cannot see how the data was collected, but my assumption is that player identity it was through an email survey form based on the hashed emails, and then some other program likely used that identity data to track hours of play per player. The most important variable here is the played hours, as it is what will be used to represent the amount of data a player contributes to the study (more hours = more data). The other variables will be used to predict the hours played via a regression model.

In [5]:
players <- read_csv("data/players.csv")
head(players)

Rows: 196 Columns: 7
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (4): experience, hashedEmail, name, gender
dbl (2): played_hours, Age
lgl (1): subscribe

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


experience,subscribe,hashedEmail,played_hours,name,gender,Age
<chr>,<lgl>,<chr>,<dbl>,<chr>,<chr>,<dbl>
Pro,TRUE,f6daba428a5e19a3d47574858c13550499be23603422e6a0ee9728f8b53e192d,30.3,Morgan,Male,9
Veteran,TRUE,f3c813577c458ba0dfef80996f8f32c93b6e8af1fa939732842f2312358a88e9,3.8,Christian,Male,17
Veteran,FALSE,b674dd7ee0d24096d1c019615ce4d12b20fcbff12d79d3c5a9d2118eb7ccbb28,0.0,Blake,Male,17
Amateur,TRUE,23fe711e0e3b77f1da7aa221ab1192afe21648d47d2b4fa7a5a659ff443a0eb5,0.7,Flora,Female,21
Regular,TRUE,7dc01f10bf20671ecfccdac23812b1b415acd42c2147cb0af4d48fcce2420f3e,0.1,Kylie,Male,21
Amateur,TRUE,f58aad5996a435f16b0284a3b267f973f9af99e7a89bee0430055a44fa92f977,0.0,Adrian,Female,17


In [13]:
summary_stats <- summarize(players, 
                           avg_played_hours = mean(played_hours, na.rm = TRUE),
                           avg_age = mean(Age, na.rm = TRUE))
summary_stats

avg_played_hours,avg_age
<dbl>,<dbl>
5.845918,21.13918
